# 02_feature_engineering.ipynb
**Mục tiêu:** Nhằm chuyển đổi dữ liệu AirBnB đã được khảo sát trong giai đoạn EDA thành tập dữ liệu đặc trưng phù hợp cho bài toán dự báo listing tại Bangkok. Bao gồm:
- Xây dựng đặc trưng mới có ý nghĩa.
- Lựa chọn các biến đầu vào cần thiết. 

# Thư viện

In [9]:
import warnings
warnings.simplefilter('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import ast

# Dữ liệu

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.motherduck import connect_motherduck
from utils.sql import query_dataframe

connection = connect_motherduck(read_only=True)

In [3]:
# Dữ liệu listings
listings_sil = query_dataframe(
    connection,
    """
    select
        -- location
        neighbourhood,

        -- listing attributes
        property_type,
        room_type,
        accommodates,
        bathrooms,
        bedrooms,
        beds,
        -- pricing rules
        price,
        minimum_nights,
        maximum_nights,
        instant_bookable,

        -- host attributes
        host_response_time,
        host_acceptance_rate,
        host_is_superhost,
        host_listings_count,
        host_total_listings_count,
        calculated_host_listings_count,

        -- availability"
        availability_30,
        availability_60,
        availability_90,
        availability_365,

        -- reviews
        number_of_reviews,
        number_of_reviews_ltm,
        reviews_per_month,

        -- review scores
        review_scores_rating,
        review_scores_accuracy,
        review_scores_cleanliness,
        review_scores_checkin,
        review_scores_communication,
        review_scores_location,
        review_scores_value,

        amenities

    from airbnb_analytics.silver.silver_listings
    order by listing_id
    """
)

listings_sil.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,neighbourhood,property_type,room_type,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,maximum_nights,...,number_of_reviews_ltm,reviews_per_month,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,amenities
0,Ratchathewi,Entire condo,Entire home/apt,2,1.5,1.0,1.0,1595.0,15,240,...,0,0.40,4.86,4.95,4.82,4.97,4.91,4.66,4.75,"[""Free parking on premises"", ""Dryer"", ""Hot wat..."
1,Bang Na,Private room in rental unit,Private room,2,1.0,NaN,NaN,NaN,1,730,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[""Essentials"", ""Heating"", ""Free parking on pre..."
2,Bang Kapi,Private room in rental unit,Private room,2,1.0,1.0,NaN,NaN,60,730,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[]
3,Don Mueang,Entire home,Entire home/apt,1,NaN,4.0,1.0,4188.0,3,730,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[""Free parking on premises"", ""Kitchen"", ""Pets ..."
4,Rat Burana,Private room in rental unit,Private room,2,1.0,1.0,1.0,1450.0,14,365,...,0,0.01,5.00,5.00,5.00,5.00,5.00,5.00,5.00,"[""Free parking on premises"", ""Drying rack for ..."


# Loại bỏ các đặc trưng ban đầu

**Mục tiêu:** Loại bỏ một số đặc trưng khó sử dụng và không cần thiết cho mô hình. Gồm:
- `listing_id`
- `listing_name`
- `host_id`
- `host_name`
- `host_since`
- `host_location`
- `host_response_rate`
- `host_identity_verified`
- `latitude`
- `longtitude`
- `amenities`
- `last_scraped`
- `last_review`

# Feature Engineering

In [4]:
listings_sil.info()

<class 'pandas.DataFrame'>
RangeIndex: 28806 entries, 0 to 28805
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   neighbourhood                   28806 non-null  str    
 1   property_type                   28806 non-null  str    
 2   room_type                       28806 non-null  str    
 3   accommodates                    28806 non-null  int32  
 4   bathrooms                       28655 non-null  float64
 5   bedrooms                        27502 non-null  float64
 6   beds                            23224 non-null  float64
 7   price                           23273 non-null  float64
 8   minimum_nights                  28806 non-null  int32  
 9   maximum_nights                  28806 non-null  int32  
 10  instant_bookable                28806 non-null  bool   
 11  host_response_time              28797 non-null  str    
 12  host_acceptance_rate            24527 non-n

In [7]:
listings_features = listings_sil.copy()

print("Kích thước ban đầu:", listings_features.shape)

Kích thước ban đầu: (28806, 32)


In [8]:
listings_features = listings_features.dropna(subset=["price"]).copy()

listings_features["log_price"] = np.log1p(listings_features["price"])

print("Kích thước sau khi loại price thiếu:", listings_features.shape)

Kích thước sau khi loại price thiếu: (23273, 33)


In [10]:
listings_features["has_reviews"] = (
    listings_features["number_of_reviews"] > 0
).astype("int8")

In [11]:
def count_amenities(value):
    if pd.isna(value):
        return np.nan

    # Trường hợp dữ liệu là chuỗi biểu diễn list
    if isinstance(value, str):
        value = value.strip()

        if not value:
            return 0

        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, (list, tuple, set)):
                return len(parsed)

        except (ValueError, SyntaxError):
            pass
    return np.nan

In [12]:
listings_features["amenities_count"] = (
    listings_features["amenities"]
    .apply(count_amenities)
    .astype("Int32")
)

In [ ]:
listings_features = listings_features.drop(
    columns=["amenities", "price"]
)

In [20]:
column_summary = pd.DataFrame({
    "column": listings_features.columns,
    "dtype": listings_features.dtypes.astype(str).values,
    "non_null_count": listings_features.notna().sum().values,
    "missing_count": listings_features.isna().sum().values,
    "missing_rate": (
        listings_features.isna().mean().mul(100).round(2).values
    ),
    "unique_count": listings_features.nunique(dropna=True).values
})

column_summary = column_summary.sort_values(
    by=["missing_rate", "column"],
    ascending=[False, True]
).reset_index(drop=True)
print("============= TỔNG HỢP DỮ LIỆU SAU KHI TẠO ĐẶC TRƯNG =============")
column_summary

============= TỔNG HỢP DỮ LIỆU SAU KHI TẠO ĐẶC TRƯNG =============


,column,dtype,non_null_count,missing_count,missing_rate,unique_count
0,review_scores_value,float64,16113,7160,30.77,138
1,review_scores_accuracy,float64,16115,7158,30.76,135
2,review_scores_checkin,float64,16114,7159,30.76,134
3,review_scores_cleanliness,float64,16115,7158,30.76,147
4,review_scores_communication,float64,16114,7159,30.76,127
5,review_scores_location,float64,16114,7159,30.76,149
6,review_scores_rating,float64,16115,7158,30.76,141
7,reviews_per_month,float64,16115,7158,30.76,671
8,host_acceptance_rate,float64,21551,1722,7.40,98
9,host_is_superhost,boolean,21799,1474,6.33,2


**Nhận xét:**
Dữ liệu được giữ nguyên tối đa và chỉ bổ sung các đặc trưng mới có ý nghĩa nghiệp vụ như has_reviews, amenities_count. Các bước xử lý phụ thuộc vào phân phối dữ liệu như điền giá trị thiếu bằng median hoặc mode, mã hóa biến phân loại và chuẩn hóa dữ liệu được thực hiện trong pipeline của notebook Modeling, sau khi chia train–test, nhằm tránh rò rỉ thông tin từ tập kiểm tra và bảo đảm kết quả đánh giá mô hình khách quan.